In [ ]:
# Computer Vision
import cv2
import mediapipe as mp

# Data processing
import numpy as np
import json
from pathlib import Path

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Dataset split
from sklearn.model_selection import train_test_split

# Progress bar
from tqdm import tqdm

In [2]:
# Sử dụng r"" để tránh lỗi ký tự \ trong Windows
DATASET_PATH = Path(r"C:\Users\Admin\Desktop\project2\dataset")

VIDEO_PATH = DATASET_PATH / "videos"
JSON_PATH = DATASET_PATH / "nslt_100.json"
KEYPOINT_PATH = DATASET_PATH / "keypoints"

# Tạo thư mục chứa keypoints nếu chưa có
KEYPOINT_PATH.mkdir(exist_ok=True, parents=True)

print("Thư mục chứa video gốc:", VIDEO_PATH)
print("File chứa metadata:", JSON_PATH)
print("Thư mục lưu keypoints:", KEYPOINT_PATH)

Thư mục chứa video gốc: C:\Users\Admin\Desktop\project2\dataset\videos
File chứa metadata: C:\Users\Admin\Desktop\project2\dataset\nslt_100.json
Thư mục lưu keypoints: C:\Users\Admin\Desktop\project2\dataset\keypoints


In [3]:
with open(JSON_PATH, "r") as f:
    metadata = json.load(f)

print("Total videos:", len(metadata))

# xem thử 1 sample
sample_key = list(metadata.keys())[0]
print("Example:", sample_key)
print(metadata[sample_key])

Total videos: 2038
Example: 05237
{'subset': 'train', 'action': [77, 1, 55]}


In [4]:
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5
)

In [5]:
def extract_keypoints_v2(results):
    keypoints = np.zeros(126)
    if results.multi_hand_landmarks:
        coords = []
        for hand_landmarks in results.multi_hand_landmarks:
            wrist = hand_landmarks.landmark[0]
            # Tính khoảng cách từ cổ tay đến gốc ngón giữa (điểm 9) để làm thước đo
            m_joint = hand_landmarks.landmark[9]
            scale = np.sqrt((m_joint.x - wrist.x)**2 + (m_joint.y - wrist.y)**2) + 1e-6

            for lm in hand_landmarks.landmark:
                # Trừ gốc cổ tay VÀ chia cho scale để cố định kích thước bàn tay
                coords.extend([
                    (lm.x - wrist.x) / scale, 
                    (lm.y - wrist.y) / scale, 
                    (lm.z - wrist.z) / scale
                ])
        coords = coords[:126]
        keypoints[:len(coords)] = coords
    return keypoints

In [12]:
MAX_FRAMES = 120

# 1. Hàm đệm để gò ép mọi video về chuẩn 120 frames
def pad_sequence(seq):
    if len(seq) > MAX_FRAMES:
        return seq[:MAX_FRAMES]
    padding = np.zeros((MAX_FRAMES - len(seq), 126))
    return np.vstack((seq, padding))

# 2. Khởi tạo mảng lưu dữ liệu xịn
X_list = []
y_list = []

print("🚀 Bắt đầu cày dữ liệu chuẩn hóa cho 100 từ vựng...")

for video_id, info in tqdm(metadata.items()):
    label = info["action"][0]
    
    # BỘ LỌC TỐI THƯỢNG: Chỉ lấy 100 từ đầu tiên (0 -> 99)
    if label >= 10: 
        continue
        
    video_file = VIDEO_PATH / f"{video_id}.mp4"
    if not video_file.exists():
        continue

    cap = cv2.VideoCapture(str(video_file))
    sequence = []

    while True:
        ret, frame = cap.read()
        if not ret: break

        # BƯỚC BẮT BUỘC: Đưa hình ảnh qua MediaPipe để lấy tọa độ
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False                  
        results = hands.process(frame_rgb)
        
        # Đưa 'results' vào hàm chuẩn hóa bạn vừa viết ở Cell 5
        keypoints = extract_keypoints_v2(results)
        sequence.append(keypoints)

    cap.release()
    
    # Chỉ xử lý nếu video thực sự có frame
    if len(sequence) > 0:
        # Cắt/Ghép cho chuẩn 120 frames
        seq_padded = pad_sequence(sequence)
        
        # Lưu file .npy để backup vào ổ cứng
        np.save(KEYPOINT_PATH / f"{video_id}.npy", seq_padded)
        
        # Đẩy thẳng vào mảng RAM để lát nữa train luôn, không cần load lại
        X_list.append(seq_padded)
        y_list.append(label)

# Chuyển sang định dạng Numpy Array
X = np.array(X_list)
y = np.array(y_list)

print(f"\n✅ Xong! Kích thước X: {X.shape}")
print(f"✅ Kích thước y: {y.shape}")

🚀 Bắt đầu cày dữ liệu chuẩn hóa cho 100 từ vựng...


100%|██████████| 2038/2038 [03:06<00:00, 10.91it/s]


✅ Xong! Kích thước X: (119, 120, 126)
✅ Kích thước y: (119,)


In [13]:
NUM_CLASSES = 10

y = to_categorical(y, NUM_CLASSES)

print(y.shape)

(119, 10)


In [14]:
# Lưu ý: Nhớ truyền vào y_categorical (nhãn đã chuyển sang dạng số 0 và 1 ở cell trước)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify = y 
)

print("Dữ liệu huấn luyện (Train): X =", X_train.shape, "| y =", y_train.shape)
print("Dữ liệu kiểm thử (Test)  : X =", X_test.shape, "| y =", y_test.shape)

Dữ liệu huấn luyện (Train): X = (95, 120, 126) | y = (95, 10)
Dữ liệu kiểm thử (Test)  : X = (24, 120, 126) | y = (24, 10)


In [15]:
model = Sequential()

model.add(LSTM(128, return_sequences=True, input_shape=(120,126)))
model.add(Dropout(0.3))

model.add(LSTM(128))
model.add(Dropout(0.3))

model.add(Dense(128, activation="relu"))

model.add(Dense(10, activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 120, 128)          130560    
                                                                 
 dropout (Dropout)           (None, 120, 128)          0         
                                                                 
 lstm_1 (LSTM)               (None, 128)               131584    
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense (Dense)               (None, 128)               16512     
                                                                 
 dense_1 (Dense)             (None, 10)                1290      
                                                                 
Total params: 279,946
Trainable params: 279,946
Non-trai

In [16]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=8,
    validation_data=(X_test, y_test)
)

Epoch 1/50
12/12 [==============================] - 3s 97ms/step - loss: 2.3023 - accuracy: 0.0421 - val_loss: 2.2863 - val_accuracy: 0.1250
Epoch 2/50
12/12 [==============================] - 1s 52ms/step - loss: 2.2719 - accuracy: 0.0947 - val_loss: 2.1950 - val_accuracy: 0.1250
Epoch 3/50
12/12 [==============================] - 1s 51ms/step - loss: 2.2305 - accuracy: 0.1053 - val_loss: 2.0704 - val_accuracy: 0.1667
Epoch 4/50
12/12 [==============================] - 1s 52ms/step - loss: 2.0576 - accuracy: 0.1684 - val_loss: 2.0049 - val_accuracy: 0.2500
Epoch 5/50
12/12 [==============================] - 1s 54ms/step - loss: 2.0561 - accuracy: 0.1474 - val_loss: 1.9714 - val_accuracy: 0.2083
Epoch 6/50
12/12 [==============================] - 1s 56ms/step - loss: 2.0287 - accuracy: 0.1368 - val_loss: 2.1078 - val_accuracy: 0.2083
Epoch 7/50
12/12 [==============================] - 1s 54ms/step - loss: 1.9729 - accuracy: 0.1579 - val_loss: 2.1510 - val_accuracy: 0.2083
Epoch 8/50
12

In [17]:
model.save("sign_language_model.h5")

print("Model saved!")

Model saved!


In [18]:
seq = np.load(KEYPOINT_PATH / f"{video_id}.npy")

seq = pad_sequence(seq)

seq = np.expand_dims(seq, axis=0)

pred = model.predict(seq)

label = np.argmax(pred)

print("Predicted label:", label)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Admin\\Desktop\\project2\\dataset\\keypoints\\28202.npy'

In [ ]:
# import cv2
# import numpy as np
# import mediapipe as mp

# # 1. Khai báo thêm công cụ vẽ của MediaPipe
# mp_drawing = mp.solutions.drawing_utils
# mp_drawing_styles = mp.solutions.drawing_styles
# mp_hands = mp.solutions.hands

# # Tạo một hàm trích xuất mới (Nhận trực tiếp 'results' thay vì 'frame' để tối ưu tốc độ)
# def extract_keypoints_from_results(results):
#     keypoints = np.zeros(126)
#     if results.multi_hand_landmarks:
#         coords = []
#         for hand_landmarks in results.multi_hand_landmarks:
#             for lm in hand_landmarks.landmark:
#                 coords.extend([lm.x, lm.y, lm.z])
#         coords = coords[:126]
#         keypoints[:len(coords)] = coords
#     return keypoints

# # 2. Các biến trạng thái ban đầu
# sequence = []          
# current_word = "..."   
# confidence = 0.0       
# threshold = 0.3        

# # Mở Webcam
# cap = cv2.VideoCapture(0)

# # Khởi tạo MediaPipe Hands trực tiếp trong block này
# with mp_hands.Hands(
#     model_complexity=0, # Dùng model nhẹ nhất cho Webcam để đỡ giật
#     min_detection_confidence=0.5,
#     min_tracking_confidence=0.5) as hands:

#     print("🎥 Đang khởi động Webcam có khung xương... Bấm 'q' để thoát.")

#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break
            
#         # Lật ngược khung hình như soi gương
#         frame = cv2.flip(frame, 1)
        
#         # Chuyển màu để MediaPipe xử lý
#         frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         frame_rgb.flags.writeable = False
        
#         # BỘ NÃO MEDIAPIPE HOẠT ĐỘNG (Chỉ chạy 1 lần duy nhất)
#         results = hands.process(frame_rgb)
        
#         frame_rgb.flags.writeable = True
        
#         # --- BƯỚC MỚI: VẼ KHUNG XƯƠNG LÊN MÀN HÌNH ---
#         if results.multi_hand_landmarks:
#             for hand_landmarks in results.multi_hand_landmarks:
#                 mp_drawing.draw_landmarks(
#                     frame, # Vẽ trực tiếp lên frame hiển thị
#                     hand_landmarks,
#                     mp_hands.HAND_CONNECTIONS,
#                     mp_drawing_styles.get_default_hand_landmarks_style(),
#                     mp_drawing_styles.get_default_hand_connections_style())
#         # ---------------------------------------------

#         # 3. Trích xuất mảng 126 số từ results để đưa cho LSTM
#         keypoints = extract_keypoints_from_results(results)
        
#         # 4. Đưa vào băng chuyền (Sliding Window)
#         sequence.append(keypoints)
#         sequence = sequence[-MAX_FRAMES:]
        
#         # 5. Bắt đầu dự đoán khi đã tích lũy đủ số frames
#         if len(sequence) == MAX_FRAMES:
#             input_data = np.expand_dims(sequence, axis=0)
#             res = model.predict(input_data, verbose=0)[0]
            
#             best_idx = np.argmax(res)
#             confidence = res[best_idx]
            
#             if confidence > threshold:
#                 current_word = label_encoder.inverse_transform([best_idx])[0]
                
#         # 6. Vẽ giao diện HUD (Bảng thông báo)
#         cv2.rectangle(frame, (0, 0), (640, 50), (245, 117, 16), -1) # Nền cam cho nổi bật
#         display_text = f"AI: {current_word.upper()} ({confidence*100:.1f}%)"
#         cv2.putText(frame, display_text, (15, 35), 
#                     cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
#         # Hiển thị
#         cv2.imshow('Sign Language AI - Realtime', frame)
        
#         # Thoát
#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# # Dọn dẹp
# cap.release()
# cv2.destroyAllWindows()

🎥 Đang khởi động Webcam có khung xương... Bấm 'q' để thoát.


In [7]:
import random
import cv2

# 1. TẢI TỪ ĐIỂN DỊCH SỐ THÀNH CHỮ (MAPPING)
class_mapping = {}
# Thay đổi tên file này nếu file từ điển của bạn tên khác nhé
class_list_file = DATASET_PATH / "wlasl_class_list.txt" 

if class_list_file.exists():
    with open(class_list_file, "r") as f:
        for line in f:
            parts = line.strip().split() # Cắt theo dấu tab hoặc khoảng trắng
            if len(parts) >= 2:
                class_id = int(parts[0])
                word = " ".join(parts[1:])
                class_mapping[class_id] = word.upper()
else:
    print("⚠️ Cảnh báo: Không tìm thấy file wlasl_class_list.txt!")
    print("⚠️ Máy sẽ chỉ hiển thị Class ID, hoặc bạn tự tạo dictionary bằng tay nhé.")

def explore_dataset_videos():
    available_videos = []
    for vid_id, info in metadata.items():
        vid_file = VIDEO_PATH / f"{vid_id}.mp4"
        if vid_file.exists():
            label = info["action"][0] 
            available_videos.append((vid_id, label, str(vid_file)))
            
    if not available_videos:
        print("❌ Không tìm thấy video nào trong thư mục!")
        return

    print("=" * 50)
    print("📺 TRÌNH PHÁT VIDEO WLASL (CÓ VIETSUB)")
    print("👉 Bấm phím 'n' để chuyển video.")
    print("👉 Bấm phím 'q' để THOÁT.")
    print("=" * 50)

    while True:
        random_vid_id, word_id, video_path = random.choice(available_videos)
        
        # 2. DỊCH SỐ THÀNH CHỮ TỪ TỪ ĐIỂN
        # Nếu không tìm thấy trong từ điển, nó sẽ hiện "UNKNOWN"
        word_name = class_mapping.get(word_id, "???") 
        
        # Gộp cả Class ID và Chữ để in ra màn hình
        word_str = f"Class {word_id}: {word_name}"
        
        print(f"🎬 Đang phát: {word_str} (ID video: {random_vid_id})")
        
        cap = cv2.VideoCapture(video_path)
        quit_app = False
        next_vid = False

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
                
            # In dòng chữ "Class 77: DEAF" lên màn hình video
            cv2.putText(frame, word_str, (20, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3) # Chữ màu vàng cho dễ nhìn
                
            cv2.imshow("WLASL Dictionary - Nhan 'n' de tiep tuc", frame)
            
            key = cv2.waitKey(30) & 0xFF
            if key == ord('q'):
                quit_app = True
                break
            elif key == ord('n'):
                next_vid = True
                break

        cap.release()
        if quit_app:
            print("Đã đóng trình phát video.")
            break

    cv2.destroyAllWindows()

# Chạy trình phát
explore_dataset_videos()

📺 TRÌNH PHÁT VIDEO WLASL (CÓ VIETSUB)
👉 Bấm phím 'n' để chuyển video.
👉 Bấm phím 'q' để THOÁT.
🎬 Đang phát: Class 88: LETTER (ID video: 66038)
🎬 Đang phát: Class 35: CAN (ID video: 08935)
🎬 Đang phát: Class 12: HELP (ID video: 27213)
🎬 Đang phát: Class 28: TABLE (ID video: 56563)
🎬 Đang phát: Class 18: ALL (ID video: 01986)
🎬 Đang phát: Class 22: HOT (ID video: 28116)
🎬 Đang phát: Class 81: CHEAT (ID video: 65342)
🎬 Đang phát: Class 86: HOW (ID video: 69370)
🎬 Đang phát: Class 30: WHAT (ID video: 69531)
Đã đóng trình phát video.


In [8]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# 1. LOAD MODEL XỊN (Bản đã chuẩn hóa 100 từ)
print("⏳ Đang tải bộ não AI chuẩn hóa...")
model = load_model("sign_language_model.h5") 

# 2. LOAD TỪ ĐIỂN (Để hiện chữ tiếng Anh thay vì Class ID)
class_mapping = {}
try:
    with open(r"C:\Users\Admin\Desktop\project2\dataset\wlasl_class_list.txt", "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                class_mapping[int(parts[0])] = " ".join(parts[1:]).upper()
except:
    print("⚠️ Không tìm thấy file từ điển, sẽ hiển thị Class ID.")

# 3. HÀM TRÍCH XUẤT CHUẨN HÓA (Bắt buộc phải giống lúc Train)
def extract_keypoints_normalized(results):
    keypoints = np.zeros(126)
    if results.multi_hand_landmarks:
        coords = []
        for hand_landmarks in results.multi_hand_landmarks:
            wrist = hand_landmarks.landmark[0] # Lấy cổ tay làm gốc
            for lm in hand_landmarks.landmark:
                coords.extend([lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z])
        coords = coords[:126]
        keypoints[:len(coords)] = coords
    return keypoints

# 4. KHỞI TẠO WEBCAM
MAX_FRAMES = 120
sequence = []
current_word = "..."
confidence = 0.0

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(0)

with mp_hands.Hands(model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    print("🎥 AI READY! Hãy múa trước camera (Bấm 'q' để thoát)")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        
        # Vẽ xương tay cho đẹp và dễ căn chỉnh
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        # Trích xuất chuẩn hóa và đưa vào chuỗi 120 frames
        keypoints = extract_keypoints_normalized(results)
        sequence.append(keypoints)
        sequence = sequence[-MAX_FRAMES:]
        
        if len(sequence) == MAX_FRAMES:
            input_data = np.expand_dims(sequence, axis=0)
            res = model.predict(input_data, verbose=0)[0]
            idx = np.argmax(res)
            confidence = res[idx]
            
            if confidence > 0.00001: # Ngưỡng tin cậy
                current_word = class_mapping.get(idx, f"Class {idx}")

        # GIAO DIỆN HIỂN THỊ (HUD)
        cv2.rectangle(frame, (0, 0), (640, 60), (245, 117, 16), -1)
        display_text = f"{current_word} ({confidence*100:.1f}%)"
        cv2.putText(frame, display_text, (15, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2, cv2.LINE_AA)
        
        cv2.imshow('WLASL Recognition', frame)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

⏳ Đang tải bộ não AI chuẩn hóa...
🎥 AI READY! Hãy múa trước camera (Bấm 'q' để thoát)
